# Shot Embedding — SigLIP clip-vector pooling (Kaggle)

Notebook này chỉ đọc artifact Clip và bảng Shot từ Kaggle Dataset, rồi ghi một full rebuild artifact vào `/kaggle/working`. Nó không mở video, không load/inference SigLIP, không kết nối hay import PostgreSQL/FAISS production, không upload và không chạy caption/ASR.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import time
import tracemalloc
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import faiss

INPUT_DIR = Path("/kaggle/input/btc-clip-embedding-output")
OUTPUT_DIR = Path("/kaggle/working/shot_embedding_output")
SHOT_FILE = INPUT_DIR / "shot.csv"  # accepts shots.csv fallback
CLIP_ARTIFACT_ROOT = INPUT_DIR / "clip_embedding_output"
VIDEO_START = 0
VIDEO_END = None
ROWS_PER_SHARD = 25_000
INDEX_VERSION = 1
RUN_ID = None
ALLOW_CHECKPOINT_OVERRIDE = False

MODEL_ID = "google/siglip2-base-patch16-224"
MODEL_REVISION = "75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2"
SAMPLING_VERSION = "uniform_midpoint_16_v1"
CLIP_POOLING_VERSION = "masked_mean_v1"
AGGREGATION_VERSION = "coverage_weighted_mean_v1"

def discover_shot_input() -> None:
    global INPUT_DIR, SHOT_FILE, CLIP_ARTIFACT_ROOT
    if SHOT_FILE.is_file() and (CLIP_ARTIFACT_ROOT / "manifest.json").is_file(): return
    roots = [INPUT_DIR, Path("/kaggle/input")]
    manifests = sorted({candidate for root in roots if root.is_dir() for candidate in root.rglob("manifest.json")})
    for manifest in manifests:
        try: payload=json.loads(manifest.read_text(encoding="utf-8"))
        except (OSError,json.JSONDecodeError): continue
        if str(payload.get("entity_type")) != "clip": continue
        candidates=[manifest.parent/"shot.csv",manifest.parent/"shots.csv",manifest.parent.parent/"shot.csv",manifest.parent.parent/"shots.csv"]
        shot=next((candidate for candidate in candidates if candidate.is_file()),None)
        if shot is not None:
            INPUT_DIR,SHOT_FILE,CLIP_ARTIFACT_ROOT=shot.parent,shot,manifest.parent; print("Using Shot input:",manifest.parent); return
    raise FileNotFoundError("Drop a Clip artifact folder plus shot.csv/shots.csv into the notebook input")

discover_shot_input()

if not SHOT_FILE.is_file():
    SHOT_FILE = INPUT_DIR / "shots.csv"
if not SHOT_FILE.is_file():
    raise FileNotFoundError(f"Missing shot.csv/shots.csv under {INPUT_DIR}")
if not (CLIP_ARTIFACT_ROOT / "manifest.json").is_file():
    raise FileNotFoundError(f"Missing Clip manifest: {CLIP_ARTIFACT_ROOT / 'manifest.json'}")
if ROWS_PER_SHARD <= 0:
    raise ValueError("ROWS_PER_SHARD must be positive")
print("numpy", np.__version__, "pandas", pd.__version__, "faiss", faiss.__version__)
print("CPU pooling is the default; no model or video is loaded.")

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def manifest_paths(manifest: dict[str, Any], key: str, fallback_glob: str) -> list[str]:
    paths = manifest.get(key) or []
    if isinstance(paths, dict):
        paths = list(paths.values())
    if not paths:
        paths = [str(path.relative_to(CLIP_ARTIFACT_ROOT)) for path in sorted(CLIP_ARTIFACT_ROOT.glob(fallback_glob))]
    return [str(path) for path in paths]

def resolve_artifact_path(value: str) -> Path:
    path = Path(value)
    return path if path.is_absolute() else CLIP_ARTIFACT_ROOT / path

def require_manifest_field(manifest: dict[str, Any], *names: str) -> Any:
    for name in names:
        if manifest.get(name) is not None:
            return manifest[name]
    raise ValueError(f"Clip manifest lacks required field; expected one of {names}")

def normalized_l2(vector: np.ndarray) -> np.ndarray:
    result = np.asarray(vector, dtype=np.float32)
    if result.ndim != 1 or not np.isfinite(result).all():
        raise ValueError("vector is not a finite 1-D array")
    norm = float(np.linalg.norm(result))
    if not np.isfinite(norm) or norm <= 0.0:
        raise ValueError("vector has zero or invalid norm")
    return np.ascontiguousarray(result / norm, dtype=np.float32)

def coverage_weights(intervals: list[tuple[int, int]]) -> np.ndarray:
    """Backend-equivalent interval coverage allocation for overlapping clips."""
    if not intervals:
        return np.empty(0, dtype=np.float64)
    boundaries = sorted({point for start, end in intervals for point in (start, end)})
    weights = np.zeros(len(intervals), dtype=np.float64)
    for left, right in zip(boundaries, boundaries[1:]):
        covered = [index for index, (start, end) in enumerate(intervals) if start <= left and end >= right]
        if right > left and covered:
            weights[covered] += (right - left) / len(covered)
    return weights

def has_overlap(intervals: list[tuple[int, int]]) -> bool:
    previous_end: int | None = None
    for start, end in sorted(intervals):
        if previous_end is not None and start < previous_end:
            return True
        previous_end = max(previous_end or end, end)
    return False

def pool_shot_vectors(vectors: np.ndarray, intervals: list[tuple[int, int]]) -> tuple[np.ndarray, np.ndarray]:
    if vectors.ndim != 2 or len(vectors) != len(intervals) or not len(vectors):
        raise ValueError("vectors and intervals must be non-empty and aligned")
    if not np.isfinite(vectors).all():
        raise ValueError("clip vector contains non-finite values")
    if len(vectors) == 1:
        return normalized_l2(vectors[0]), np.array([1.0], dtype=np.float64)
    weights = coverage_weights(intervals) if has_overlap(intervals) else np.asarray([end - start for start, end in intervals], dtype=np.float64)
    if not np.isfinite(weights).all() or float(weights.sum()) <= 0:
        raise ValueError("coverage weights are invalid")
    return normalized_l2(np.average(vectors, axis=0, weights=weights)), weights

def sql_literal(value: Any) -> str:
    if isinstance(value, int):
        return str(value)
    return "'" + str(value).replace("'", "''") + "'"


In [ ]:
# Fail-fast compatibility gate: validation completes before any Shot output is created.
clip_manifest_path = CLIP_ARTIFACT_ROOT / "manifest.json"
clip_manifest = json.loads(clip_manifest_path.read_text(encoding="utf-8"))
if str(require_manifest_field(clip_manifest, "entity_type")) != "clip":
    raise ValueError("Shot aggregation requires entity_type='clip'")
if require_manifest_field(clip_manifest, "normalized") is not True:
    raise ValueError("Clip manifest must declare normalized=true")
manifest_model_id = require_manifest_field(clip_manifest, "model_id", "model_name")
if manifest_model_id != MODEL_ID:
    raise ValueError(f"Unexpected model: {manifest_model_id!r}")
if require_manifest_field(clip_manifest, "model_revision") != MODEL_REVISION:
    raise ValueError("Clip model revision differs from required revision")
sampling = str(require_manifest_field(clip_manifest, "sampling_version", "sampling_strategy"))
if "uniform" not in sampling.lower() or "16" not in sampling:
    raise ValueError(f"Clip sampling is not 16 uniform midpoint: {sampling}")
pooling = str(require_manifest_field(clip_manifest, "aggregation_version", "pooling_version"))
if pooling != CLIP_POOLING_VERSION:
    raise ValueError(f"Clip pooling must be {CLIP_POOLING_VERSION}, got {pooling}")
dimension = int(require_manifest_field(clip_manifest, "dimension"))
vector_paths = manifest_paths(clip_manifest, "vector_shards", "vectors/part-*.npy")
metadata_paths = manifest_paths(clip_manifest, "metadata_shards", "metadata/part-*.parquet")
if not vector_paths or not metadata_paths:
    raise ValueError("Clip artifact must contain vector and metadata shards")
for relative_path, expected in (clip_manifest.get("checksums") or {}).items():
    candidate = resolve_artifact_path(relative_path)
    if candidate.is_file() and sha256_file(candidate) != expected:
        raise ValueError(f"Clip artifact checksum mismatch: {relative_path}")

shots = pd.read_csv(SHOT_FILE)
required_shot_columns = ['shot_id', 'video_id', 'shot_index', 'start_ms', 'end_ms', 'start_frame_idx', 'end_frame_idx']
if missing := set(required_shot_columns).difference(shots.columns):
    raise ValueError(f"Shot table missing columns: {sorted(missing)}")
shots = shots[required_shot_columns].copy()
if shots[['shot_id', 'video_id']].isna().any().any() or shots.shot_id.duplicated().any():
    raise ValueError("Shot IDs/video IDs must be present and shot_id unique")
for column in ['shot_index', 'start_ms', 'end_ms', 'start_frame_idx', 'end_frame_idx']:
    shots[column] = pd.to_numeric(shots[column], errors='raise').astype('int64')
if (shots.end_ms <= shots.start_ms).any() or (shots.start_ms < 0).any():
    raise ValueError("Input Shot interval is invalid")
sorted_video_ids = sorted(shots.video_id.astype(str).unique())
selected_video_ids = set(sorted_video_ids[VIDEO_START:VIDEO_END])
shots = shots[shots.video_id.astype(str).isin(selected_video_ids)].sort_values(['video_id', 'shot_index', 'shot_id'], kind='stable').reset_index(drop=True)
shot_lookup = shots.set_index('shot_id').to_dict('index')

metadata_frames = []
for relative_path in metadata_paths:
    path = resolve_artifact_path(relative_path)
    frame = pd.read_parquet(path)
    frame['__metadata_path'] = relative_path
    metadata_frames.append(frame)
clip_metadata = pd.concat(metadata_frames, ignore_index=True)
required_clip_columns = {'faiss_id', 'clip_id', 'shot_id', 'video_id', 'start_ms', 'end_ms', 'vector_shard', 'vector_row', 'model_revision', 'dimension', 'normalized'}
if missing := required_clip_columns.difference(clip_metadata.columns):
    raise ValueError(f"Clip metadata missing columns: {sorted(missing)}")
clip_metadata = clip_metadata.copy()
if len(clip_metadata) != int(require_manifest_field(clip_manifest, 'success_count')):
    raise ValueError("Clip success metadata count differs from manifest")
for _, row in clip_metadata.iterrows():
    shot = shot_lookup.get(str(row.shot_id))
    if shot is None or str(row.video_id) != str(shot['video_id']):
        raise ValueError(f"Unknown or wrong-video clip shot_id: {row.shot_id}")
    if int(row.start_ms) < int(shot['start_ms']) or int(row.end_ms) > int(shot['end_ms']) or int(row.end_ms) <= int(row.start_ms):
        raise ValueError(f"Clip interval outside Shot: {row.clip_id}")
    if int(row.dimension) != dimension or str(row.model_revision) != MODEL_REVISION or bool(row.normalized) is not True:
        raise ValueError(f"Incompatible Clip metadata: {row.clip_id}")
vector_rows_by_shard = {}
for index, relative_path in enumerate(vector_paths):
    matrix = np.load(resolve_artifact_path(relative_path), mmap_mode='r')
    if matrix.ndim != 2 or matrix.shape[1] != dimension or not np.isfinite(matrix).all():
        raise ValueError(f"Invalid Clip vector shard: {relative_path}")
    vector_rows_by_shard[str(index)] = len(matrix)
for _, row in clip_metadata.iterrows():
    shard_key, vector_row = str(row.vector_shard), int(row.vector_row)
    if shard_key not in vector_rows_by_shard or not 0 <= vector_row < vector_rows_by_shard[shard_key]:
        raise ValueError(f"Invalid Clip vector mapping: {row.clip_id}")
print(f"Compatibility gate passed: {len(shots)} selected shots, {len(clip_metadata)} Clip vectors, D={dimension}")

In [ ]:
# Smoke/unit tests for the required pooling edge cases.
fixture = np.asarray([[1., 0.], [0., 1.], [-0.6, 0.8]], dtype=np.float32)
overlap_intervals = [(0, 10), (8, 18), (16, 26)]
overlap_weights = coverage_weights(overlap_intervals)
assert np.isclose(overlap_weights.sum(), 26.0)
coverage_vector, _ = pool_shot_vectors(fixture, overlap_intervals)
unweighted = normalized_l2(fixture.mean(axis=0))
assert not np.allclose(coverage_vector, unweighted)
duration_vector, duration_weights = pool_shot_vectors(fixture[:2], [(0, 10), (10, 30)])
assert np.allclose(duration_weights, [10., 20.]) and np.isclose(np.linalg.norm(duration_vector), 1.0)
assert 'missing-clip-shot' not in defaultdict(list)
print('Pooling tests passed (overlap coverage, duration weight, and missing-shot handling).')

In [ ]:
# Process one Shot at a time. Vectors stay memory-mapped until their owning Shot is pooled.
run_id = RUN_ID or f"shot-siglip-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
final_dir = OUTPUT_DIR / run_id
temp_dir = OUTPUT_DIR / f".{run_id}.tmp"
checkpoint_path = OUTPUT_DIR / f".{run_id}.checkpoint.json"
config_hash = hashlib.sha256(json.dumps({'input_manifest_sha256': sha256_file(clip_manifest_path), 'shot_sha256': sha256_file(SHOT_FILE), 'videos': sorted(selected_video_ids), 'rows_per_shard': ROWS_PER_SHARD, 'index_version': INDEX_VERSION, 'aggregation': AGGREGATION_VERSION}, sort_keys=True).encode()).hexdigest()
if final_dir.exists():
    raise FileExistsError(f"Output already exists (idempotency protection): {final_dir}")
if checkpoint_path.exists():
    checkpoint = json.loads(checkpoint_path.read_text())
    if checkpoint.get('config_hash') != config_hash and not ALLOW_CHECKPOINT_OVERRIDE:
        raise ValueError('Checkpoint input/config mismatch; set ALLOW_CHECKPOINT_OVERRIDE only after review')
    if checkpoint.get('resume_version') != 1 or not (temp_dir / 'vectors').is_dir() or not (temp_dir / 'metadata').is_dir():
        raise RuntimeError('Legacy or incomplete checkpoint cannot be resumed safely; review it before retrying.')
else:
    temp_dir.mkdir(parents=True, exist_ok=False)
    (temp_dir / 'vectors').mkdir(); (temp_dir / 'metadata').mkdir()
    checkpoint = {'resume_version': 1, 'config_hash': config_hash, 'completed_video_ids': [], 'next_faiss_id': 1, 'failures': [], 'state': 'started'}
    checkpoint_path.write_text(json.dumps(checkpoint, indent=2), encoding='utf-8')

mmap_shards = {str(index): np.load(resolve_artifact_path(path), mmap_mode='r') for index, path in enumerate(vector_paths)}
metadata_by_shot: dict[str, list[dict[str, Any]]] = defaultdict(list)
for row in clip_metadata.to_dict('records'):
    metadata_by_shot[str(row['shot_id'])].append(row)
for records in metadata_by_shot.values():
    records.sort(key=lambda row: (str(row['vector_shard']), int(row['vector_row']), str(row['faiss_id'])))

tracemalloc.start(); started = time.perf_counter(); pooling_seconds = 0.0
success_rows: list[dict[str, Any]] = []; failures: list[dict[str, Any]] = list(checkpoint.get('failures', [])); vector_buffer: list[np.ndarray] = []
existing_metadata_paths = sorted((temp_dir / 'metadata').glob('part-*.parquet'))
existing_metadata = pd.concat([pd.read_parquet(path) for path in existing_metadata_paths], ignore_index=True) if existing_metadata_paths else pd.DataFrame()
if not existing_metadata.empty and (existing_metadata.faiss_id.duplicated().any() or not np.array_equal(np.sort(existing_metadata.faiss_id.to_numpy(dtype=np.int64)), np.arange(1, len(existing_metadata) + 1, dtype=np.int64))):
    raise RuntimeError('Partial Shot shards have invalid FAISS IDs; cannot resume safely.')
faiss_id = len(existing_metadata) + 1; shard_index = len(list((temp_dir / 'vectors').glob('part-*.npy'))); clips_consumed = 0; completed_videos: list[str] = sorted(set(checkpoint['completed_video_ids']) | set(existing_metadata.video_id.astype(str))) if not existing_metadata.empty else list(checkpoint['completed_video_ids'])
def flush_shard() -> None:
    global shard_index, vector_buffer, success_rows
    if not vector_buffer: return
    vector_name = f'vectors/part-{shard_index:05d}.npy'; metadata_name = f'metadata/part-{shard_index:05d}.parquet'
    matrix = np.ascontiguousarray(np.vstack(vector_buffer), dtype=np.float32)
    np.save(temp_dir / vector_name, matrix)
    pd.DataFrame(success_rows).to_parquet(temp_dir / metadata_name, index=False)
    shard_index += 1; vector_buffer = []; success_rows = []

for video_id, video_shots in shots.groupby('video_id', sort=True):
    if str(video_id) in completed_videos:
        continue
    for shot in video_shots.to_dict('records'):
        records = metadata_by_shot.get(str(shot['shot_id']), [])
        if not records:
            failures.append({'shot_id': shot['shot_id'], 'video_id': shot['video_id'], 'error_message': 'no compatible clip embeddings for shot'})
            continue
        try:
            vectors = np.vstack([mmap_shards[str(row['vector_shard'])][int(row['vector_row'])] for row in records]).astype(np.float32, copy=False)
            t0 = time.perf_counter(); pooled, weights = pool_shot_vectors(vectors, [(int(row['start_ms']), int(row['end_ms'])) for row in records]); pooling_seconds += time.perf_counter() - t0
            vector_name = f'vectors/part-{shard_index:05d}.npy'
            success_rows.append({'faiss_id': faiss_id, 'shot_id': str(shot['shot_id']), 'video_id': str(shot['video_id']), 'start_ms': int(shot['start_ms']), 'end_ms': int(shot['end_ms']), 'source_embedding_ids': json.dumps([str(row['faiss_id']) for row in records]), 'source_clip_ids': json.dumps([str(row['clip_id']) for row in records]), 'coverage_weights': json.dumps([float(value) for value in weights]), 'aggregation': 'coverage_weighted_mean', 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'dimension': dimension, 'normalized': True, 'vector_shard': vector_name, 'vector_row': len(vector_buffer)})
            vector_buffer.append(pooled); faiss_id += 1; clips_consumed += len(records)
        except (ValueError, IndexError, KeyError) as error:
            failures.append({'shot_id': shot['shot_id'], 'video_id': shot['video_id'], 'error_message': str(error)})
    flush_shard()
    completed_videos.append(str(video_id))
    checkpoint = {'resume_version': 1, 'config_hash': config_hash, 'completed_video_ids': completed_videos, 'next_faiss_id': faiss_id, 'failures': failures, 'state': 'in_progress'}
    checkpoint_tmp = checkpoint_path.with_suffix('.tmp'); checkpoint_tmp.write_text(json.dumps(checkpoint, indent=2), encoding='utf-8'); checkpoint_tmp.replace(checkpoint_path)
flush_shard()
(temp_dir / 'failures.jsonl').write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in failures), encoding='utf-8')


In [ ]:
# Build the immutable index/import files, verify them, then atomically publish.
output_metadata_paths = sorted((temp_dir / 'metadata').glob('part-*.parquet'))
output_vector_paths = sorted((temp_dir / 'vectors').glob('part-*.npy'))
output_metadata = pd.concat([pd.read_parquet(path) for path in output_metadata_paths], ignore_index=True) if output_metadata_paths else pd.DataFrame()
if output_metadata.empty:
    raise RuntimeError('No successful Shot embeddings were produced; inspect the partial checkpoint and failures before retrying.')
if sum(len(pd.read_parquet(path)) for path in output_metadata_paths) != len(output_metadata):
    raise RuntimeError('Shot metadata shard count mismatch')
if output_metadata.faiss_id.duplicated().any() or output_metadata.shot_id.duplicated().any() or not np.array_equal(output_metadata.faiss_id.to_numpy(), np.arange(1, len(output_metadata) + 1)):
    raise RuntimeError('Shot metadata IDs are not unique/deterministic')
index = faiss.IndexIDMap2(faiss.IndexFlatIP(dimension))
# Add and validate one output shard at a time; never materialize all vectors in RAM.
for vector_path, metadata_path in zip(output_vector_paths, output_metadata_paths):
    shard_matrix = np.ascontiguousarray(np.load(vector_path, mmap_mode='r'), dtype=np.float32)
    shard_metadata = pd.read_parquet(metadata_path)
    if len(shard_matrix) != len(shard_metadata) or (len(shard_matrix) and (not np.isfinite(shard_matrix).all() or not np.allclose(np.linalg.norm(shard_matrix, axis=1), 1.0, atol=1e-4))):
        raise RuntimeError(f'Shot output vector validation failed: {vector_path.name}')
    expected_shard = str(vector_path.relative_to(temp_dir))
    if not (shard_metadata.vector_shard.astype(str) == expected_shard).all() or not np.array_equal(shard_metadata.vector_row.to_numpy(dtype=np.int64), np.arange(len(shard_metadata), dtype=np.int64)):
        raise RuntimeError(f'Shot output metadata mapping failed: {metadata_path.name}')
    index.add_with_ids(shard_matrix, shard_metadata.faiss_id.to_numpy(dtype=np.int64))
faiss.write_index(index, str(temp_dir / 'shot.faiss'))
if index.ntotal != len(output_metadata): raise RuntimeError('FAISS count mismatch')
if output_vector_paths:
    probe = np.ascontiguousarray(np.load(output_vector_paths[0], mmap_mode='r')[:8], dtype=np.float32)
    expected_ids = pd.read_parquet(output_metadata_paths[0]).faiss_id.to_numpy(dtype=np.int64)[:len(probe)]
    _, found_ids = index.search(probe, 1)
    if not np.array_equal(found_ids[:, 0], expected_ids): raise RuntimeError('FAISS self-search failed')
mapping_columns = ['faiss_id', 'shot_id']
mapping = output_metadata[mapping_columns].copy(); mapping['index_version'] = INDEX_VERSION; mapping['model_name'] = 'siglip2-base-patch16-224'; mapping['model_version'] = MODEL_REVISION[:50]; mapping['pooling_method'] = 'coverage_weighted_mean'
mapping[['faiss_id', 'index_version', 'shot_id', 'model_name', 'model_version', 'pooling_method']].to_csv(temp_dir / 'shot_embedding_mapping.csv', index=False)
with (temp_dir / 'insert_shot_embedding_records.sql').open('w', encoding='utf-8') as handle:
    for row in mapping.itertuples(index=False):
        values = ', '.join(sql_literal(value) for value in (row.faiss_id, row.index_version, row.shot_id, row.model_name, row.model_version, row.pooling_method))
        handle.write('INSERT INTO shotembeddingrecord (faiss_id, index_version, shot_id, model_name, model_version, pooling_method) VALUES (' + values + ') ON CONFLICT (shot_id, index_version, model_name, model_version, pooling_method) DO NOTHING;\n')
current, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
summary = {'run_id': run_id, 'shot_success_count': int(len(output_metadata)), 'shot_failure_count': int(len(failures)), 'clips_consumed': int(clips_consumed), 'io_pooling_total_seconds': time.perf_counter() - started, 'pooling_seconds': pooling_seconds, 'peak_python_ram_bytes': peak, 'gpu_usage': 'not used (CPU NumPy pooling)', 'download_output': str(final_dir)}
(temp_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
file_checksums = {str(path.relative_to(temp_dir)): sha256_file(path) for path in temp_dir.rglob('*') if path.is_file()}
manifest = {'artifact_version': 1, 'run_id': run_id, 'entity_type': 'shot', 'input_clip_artifact_manifest_sha256': sha256_file(clip_manifest_path), 'input_clip_artifact_checksums': clip_manifest.get('checksums', {}), 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'dimension': dimension, 'normalized': True, 'aggregation_version': AGGREGATION_VERSION, 'record_count': int(len(output_metadata) + len(failures)), 'success_count': int(len(output_metadata)), 'failure_count': int(len(failures)), 'vector_shards': [str(path.relative_to(temp_dir)) for path in output_vector_paths], 'metadata_shards': [str(path.relative_to(temp_dir)) for path in output_metadata_paths], 'checksums': file_checksums, 'faiss_checksum': sha256_file(temp_dir / 'shot.faiss'), 'created_at': datetime.now(timezone.utc).isoformat()}
(temp_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
shutil.move(str(temp_dir), str(final_dir)); checkpoint_path.unlink(missing_ok=True)
print(f'Download Output: {final_dir}')

In [ ]:
# Package validated Shot artifacts into one downloadable ZIP.
zip_path = Path(shutil.make_archive(str(final_dir), "zip", root_dir=final_dir))
required = [path for path in final_dir.rglob("*") if path.suffix in {".sql", ".faiss"}]
if not required:
    raise RuntimeError("ZIP packaging refused: validated SQL/FAISS files were not found")
print(f"Download ZIP: {zip_path} ({zip_path.stat().st_size / 1024**2:.1f} MiB)")
